# Segmentation d'instance des câbles et connecteurs (YOLOv8-seg)

Ce notebook est adapté au dataset **`wiring harness connectors segmen.v1i.yolov8`** (extrait depuis `wiring harness connectors segmen.v1i.yolov8.zip`).

### Ce que fait ce notebook
1. Décompresse automatiquement `wiring harness connectors segmen.v1i.yolov8.zip` si le dossier cible n'existe pas encore.
2. Audite et nettoie les fichiers de labels (`.txt`) pour garantir la compatibilité avec la segmentation (`yolov8n-seg.pt`), en supprimant les lignes à 4 coordonnées (*box-only*) si présentes et en régénérant le cache Ultralytics (`*.cache`).
3. Entraîne **YOLOv8-seg** (`yolov8n-seg.pt`) en enregistrant les résultats dans `MODELS_DIR`.

## 1. Importation des bibliothèques

In [18]:
import zipfile
from collections import Counter
from pathlib import Path
from ultralytics import YOLO

## 2. Extraction (si nécessaire) et vérification du dataset

In [19]:
# Dossier du dataset extrait et archive ZIP source
DATASET_DIR = Path("../data/processed/wiring harness connectors segmen.v1i.yolov8").resolve()
ZIP_PATH = Path("../data/raw/wiring harness connectors segmen.v1i.yolov8.zip").resolve()
DATA_YAML = DATASET_DIR / "data.yaml"

# Si le dossier n'existe pas encore mais que le ZIP est présent, on l'extrait
if not DATASET_DIR.exists() and ZIP_PATH.exists():
    print(f"Extraction de {ZIP_PATH} vers {DATASET_DIR}...")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATASET_DIR)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Fichier data.yaml introuvable : {DATA_YAML}\nVérifiez le chemin de DATASET_DIR ou ZIP_PATH."
    )

print(f"Dataset prêt : {DATASET_DIR}")
print(f"Fichier trouvé : {DATA_YAML}")

Dataset prêt : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8
Fichier trouvé : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml


## 3. Audit et nettoyage des annotations pour la segmentation (`yolo_seg`)

En segmentation YOLO, chaque instance doit être un polygone (`>= 6` coordonnées normalisées, soit un nombre pair de coordonnées après l'ID de classe). Les fichiers `.txt` vides sont conservés (images d'arrière-plan / *background*).

In [20]:
CLEAN_BOX_ONLY = True  # Retire les lignes à 4 coordonnées pour éviter la suppression de tous les segments

for split in ["train", "valid", "test"]:
    lbl_dir = DATASET_DIR / split / "labels"
    if not lbl_dir.exists():
        continue

    stats = Counter()
    for txt_file in lbl_dir.glob("*.txt"):
        lines = [ln.strip() for ln in txt_file.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            stats["empty_bg_files"] += 1
            continue

        valid_lines = []
        for ln in lines:
            parts = ln.split()
            n_coords = len(parts) - 1
            if n_coords >= 6 and n_coords % 2 == 0:
                stats["polygons"] += 1
                valid_lines.append(ln)
            elif n_coords == 4:
                stats["box_only"] += 1
                if not CLEAN_BOX_ONLY:
                    valid_lines.append(ln)
            else:
                stats["invalid"] += 1

        if CLEAN_BOX_ONLY and len(valid_lines) != len(lines):
            txt_file.write_text("\n".join(valid_lines) + ("\n" if valid_lines else ""), encoding="utf-8")

    for cache_file in (DATASET_DIR / split).glob("*.cache"):
        cache_file.unlink(missing_ok=True)

    print(f"[{split}] {dict(stats)}")

[train] {'polygons': 995, 'empty_bg_files': 12}
[valid] {'polygons': 106, 'empty_bg_files': 2}
[test] {'polygons': 25}


## 4. Chargement du modèle pré-entraîné (`yolov8n-seg.pt`)

In [21]:
model = YOLO("yolov8n-seg.pt")

## 5. Entraînement du modèle

In [22]:
MODELS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models").resolve()

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    task="segment",
    epochs=10,
    imgsz=640,
    batch=8,
    patience=5,
    workers=0,
    project=str(MODELS_DIR),
    name="harness_cable_segmentation",
    save=True,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol